# BÀI THỰC HÀNH 4: DỰ BÁO CHUỖI THỜI GIAN ĐA BIẾN (MULTIVARIATE TIME SERIES)
## MODULE CV01: QUẢN TRỊ PIPELINE, KHÁM PHÁ (EDA) & TIỀN XỬ LÝ ĐA BIẾN

> **Học phần:** Quản lý & Khai phá Chuỗi Thời gian (QLCTG) - UTH  
> **Thành viên phụ trách:** Hà Duy Hưng (Lead)  
> **Bộ dữ liệu:** Phụ tải Điện năng kết hợp Thời tiết (`load, temp, humidity, wind_speed`)  
> **Mục tiêu:** Kiểm toán dữ liệu, EDA trực quan hóa tương quan phi tuyến, chuẩn hóa bằng `StandardScaler` (chống Lookahead Bias), chia 70/15/15 và tạo 96 Lag Features ($L=24$).

In [ ]:
import sys
sys.stdout.reconfigure(encoding='utf-8')
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import trực tiếp các hàm chuẩn mực từ module CV01
import cv01_eda_and_preprocessing as cv01

print('Project Root:', cv01.PROJECT_ROOT)
print('Features    :', cv01.FEATURES)

--- 
## 1. NẠP VÀ KIỂM TOÁN DỮ LIỆU GỐC (DATA INGESTION & AUDIT)
Kiểm tra chỉ mục thời gian `DatetimeIndex`, tính liên tục tần suất giờ (`freq='h'`), giá trị khuyết thiếu và điểm dị biệt.

In [ ]:
# Đảm bảo file thô tồn tại và đọc vào
raw_path = cv01.ensure_raw_data(cv01.RAW_DATA_PATH)
df = cv01.step1_load_and_validate(raw_path)

display(df.head())
display(df.describe().round(2))

--- 
## 2. TRỰC QUAN HÓA KHÁM PHÁ (EDA VISUALIZATION)
Khảo sát hình thái chuỗi thời gian, mối quan hệ phi tuyến chữ U giữa phụ tải và nhiệt độ, ma trận tương quan Pearson và chu kỳ mùa vụ.

In [ ]:
# Kích hoạt xuất bản toàn bộ biểu đồ EDA chuẩn
cv01.step2_eda_visualize(df)

In [ ]:
# Hiển thị trực tiếp Scatter Plot mối quan hệ phi tuyến Load vs Temp
fig, ax = plt.subplots(figsize=(8, 5.5))
sc = ax.scatter(df['temp'], df['load'], c=df.index.hour, cmap='plasma', alpha=0.35, s=6)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('Giờ trong ngày (Hour 0 - 23)', fontweight='bold')
ax.set_xlabel('Nhiệt độ không khí (°C)', fontweight='bold')
ax.set_ylabel('Phụ tải Điện năng (MW)', fontweight='bold')
ax.set_title('Mối quan hệ Phi Tuyến Cong Chữ U (U-Curve): Load vs Temperature', fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.4)
plt.show()

--- 
## 3. CHUẨN HÓA & PHÂN CHIA TẬP TRAIN / VAL / TEST (ANTI-LEAKAGE)
Quy tắc vàng: **Không shuffle**. Cắt tuần tự 70% Train, 15% Val, 15% Test. `StandardScaler` CHỈ fit trên Train.

In [ ]:
train_sc, val_sc, test_sc, scaler = cv01.step3_scale_and_split(df)

print('Train Scaled Shape:', train_sc.shape)
print('Val Scaled Shape  :', val_sc.shape)
print('Test Scaled Shape :', test_sc.shape)

# Kiểm tra mean và std của tập train sau chuẩn hóa (xấp xỉ 0 và 1)
display(train_sc.agg(['mean', 'std']).round(4))

--- 
## 4. TẠO CỬA SỔ ĐỘ TRỄ ĐA BIẾN (LAG FEATURES, $L=24$)
Trích xuất $24 \times 4 = 96$ đặc trưng trễ quá khứ: $[load_{t-1 \dots t-24}, temp_{t-1 \dots t-24}, humidity_{t-1 \dots t-24}, wind_{t-1 \dots t-24}]$.

In [ ]:
train_lag = cv01.step4_create_lag_features(train_sc, lag=24)
val_lag   = cv01.step4_create_lag_features(val_sc, lag=24)
test_lag  = cv01.step4_create_lag_features(test_sc, lag=24)

print('Train with Lags Shape:', train_lag.shape)
print('Val with Lags Shape  :', val_lag.shape)
print('Test with Lags Shape :', test_lag.shape)

# Hiển thị 8 cột lag đầu tiên
display(train_lag[[c for c in train_lag.columns if 'lag' in c][:8]].head())

--- 
## 5. TỔNG KẾT VÀ BÀN GIAO SẢN PHẨM CHO CÁC MODULE SAU
- **Dữ liệu cho VAR (CV02) & LSTM (CV04):** `data/processed/train.csv`, `val.csv`, `test.csv` (tensor chuẩn hóa).
- **Dữ liệu cho XGBoost (CV03):** `data/processed/train_lag.csv`, `val_lag.csv`, `test_lag.csv` (vector phẳng 96 lags).
- **Trọng số chuẩn hóa:** `data/processed/scaler.pkl` (dùng đảo thang đo MW).
- **Báo cáo và biểu đồ:** Lưu trữ đầy đủ trong `outputs/tables/` và `outputs/figures/`.